# C11-neural-training — Practice p07 — Solution


**Type:** constrained coding · **Difficulty:** core · **Concepts:** manual-backpropagation, trained-mlp


The score derivative is \(2(\mathrm{scores}-y)/(NK)\). Matrix products reverse
the affine arrows; bias derivatives sum over axis 0 because each shared bias is
used once per example. All four gradients are formed before constructing any
updated array.


In [ ]:
import numpy as np

def one_manual_step(X, y, params, learning_rate):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    W1, b1, W2, b2 = (np.asarray(params[k], dtype=np.float64) for k in ("W1", "b1", "W2", "b2"))
    z1 = X @ W1.T + b1
    h = np.maximum(z1, 0.0)
    scores = h @ W2.T + b2
    loss_before = float(np.mean((scores - y) ** 2))
    dscores = 2.0 * (scores - y) / scores.size
    grads = {
        "W2": dscores.T @ h,
        "b2": dscores.sum(axis=0),
    }
    dh = dscores @ W2
    dz1 = dh * (z1 > 0.0)
    grads["W1"] = dz1.T @ X
    grads["b1"] = dz1.sum(axis=0)
    grads = {k: grads[k] for k in ("W1", "b1", "W2", "b2")}
    updated = {k: np.asarray(params[k], dtype=np.float64) - learning_rate * grads[k] for k in grads}
    z1_after = X @ updated["W1"].T + updated["b1"]
    scores_after = np.maximum(z1_after, 0.0) @ updated["W2"].T + updated["b2"]
    return {"loss_before": loss_before, "loss_after": float(np.mean((scores_after-y)**2)),
            "grads": grads, "updated": updated, "scores_after": scores_after}

X_p07 = np.array([[1., -1.], [2., 1.]])
y_p07 = np.array([[0.5], [1.0]])
params_p07 = {"W1": np.array([[0.4, -0.2], [0.1, 0.3]]), "b1": np.array([0.1, 0.2]),
              "W2": np.array([[0.6, -0.4]]), "b2": np.array([0.05])}
original_p07 = {k: v.copy() for k, v in params_p07.items()}
result_p07 = one_manual_step(X_p07, y_p07, params_p07, 0.1)


### Answer check


In [ ]:
z1_ref_p07=X_p07@original_p07["W1"].T+original_p07["b1"]
h_ref_p07=np.maximum(z1_ref_p07,0.0)
scores_ref_p07=h_ref_p07@original_p07["W2"].T+original_p07["b2"]
ds_ref_p07=2.0*(scores_ref_p07-y_p07)/scores_ref_p07.size
expected_p07={"W2":ds_ref_p07.T@h_ref_p07,"b2":ds_ref_p07.sum(axis=0)}
dz1_ref_p07=(ds_ref_p07@original_p07["W2"])*(z1_ref_p07>0.0)
expected_p07.update(W1=dz1_ref_p07.T@X_p07,b1=dz1_ref_p07.sum(axis=0))
assert all(np.allclose(result_p07["grads"][k],expected_p07[k],atol=1e-10,rtol=1e-9) for k in expected_p07)
assert all(result_p07["grads"][k].shape==original_p07[k].shape for k in original_p07)
assert all(np.allclose(result_p07["updated"][k],original_p07[k]-.1*expected_p07[k],atol=1e-10,rtol=1e-9) for k in original_p07)
assert all(np.array_equal(params_p07[k],original_p07[k]) for k in original_p07)
assert result_p07["scores_after"].shape==y_p07.shape
assert np.isfinite(result_p07["loss_before"]) and np.isfinite(result_p07["loss_after"])
assert any(np.linalg.norm(result_p07["updated"][k]-original_p07[k])>0 for k in original_p07)
assert result_p07["loss_after"]<result_p07["loss_before"]
